In [ ]:
!pip install --upgrade google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.4/791.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.0.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

In [ ]:
# Use the environment variable if the user doesn't provide Project ID.
import os

# fmt: off
PROJECT_ID = "PROJECT_ID"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

In [ ]:
import re
import os
import json
def format_json_response(response_text):
  print(response_text)
  response_text_temp = re.sub(r"json", "", response_text)
  response_text_temp = re.sub(r"```", "", response_text_temp)
  json_response = json.loads(response_text_temp)
  return json_response

In [ ]:
from google import genai
from google.genai import types
import base64
import os

client = genai.Client(
      vertexai=True,
      project=PROJECT_ID,
      location=LOCATION,
  )

def generate(file_uri):

  msg1_document1 = types.Part.from_uri(
      file_uri=file_uri,
      mime_type="application/pdf",
  )
  msg1_text1 = types.Part.from_text(text="""Role and Objective:
You are a meticulous Data Extraction Assistant working for the Senior Editor of a leading Hindi daily newspaper. Your task is to read the provided PDF of our newspaper and extract every single news article into a distinct, structured format. You must treat the text with absolute journalistic integrity.
Strict Constraints (Zero Tolerance Policy):
NO Hallucination: You must extract the text exactly as it appears in the document. Do not summarize, rephrase, correct grammar, or add any outside information. If a word is cut off or illegible, insert [अस्पष्ट/Unreadable] instead of guessing.
NO Mixing of Articles: Newspaper layouts are complex. Do not bleed the text of one column into an adjacent, unrelated column. Pay strict attention to headlines, dividing lines, borders, and distinct font styles to identify where one article ends and another begins.
Extraction Guidelines:
Identify Boundaries: Every article typically has a Headline (बड़ा शीर्षक), sometimes a Sub-headline (उप-शीर्षक), a Dateline/Reporter Name (स्थान/संवाददाता), and Body Text (मुख्य समाचार). Use these elements as anchors to isolate individual stories.
Follow Column Flow (Read Vertically): Newspapers are read top-to-bottom in columns, NOT strictly left-to-right across the whole page. Ensure you are following the vertical flow of the text block for a specific headline before moving to the next column.
Ignore Non-News Elements: Exclude advertisements (विज्ञापन), page headers/footers, page numbers, and purely decorative text.
Handle Jumps/Continuations: If an article ends with \"शेष पृष्ठ X पर\" (Continued on page X), mark the end of that text clearly, but do not artificially merge it with unrelated articles on the current page.
Boxed Items: Treat text enclosed in boxes (इन्फोबॉक्स/साइड स्टोरी) as separate, standalone articles unless they explicitly share the exact same main headline.
Output Format:
Provide the output in the following JSON format to ensure clean, structured separation of every article. Use Hindi script (Devanagari) for the extracted text.
[
 {
  "article_id": 1,
  "page_number": "[Insert Page Number]",
  "headline": "[Extract Main Headline Here]",
  "sub_headline": "[Extract Sub-headline if present, otherwise null]",
  "dateline_or_author": "[Extract City, Date, or Reporter Name, e.g., 'नई दिल्ली (एजेंसी)' - otherwise null]",
  "body_text": "[Extract the exact full text of the article following the correct column reading order.]"
 },
 {
  "article_id": 2,
  "page_number": "[Insert Page Number]",
  "headline": "[Extract Main Headline Here]",
  "sub_headline": "[...]",
  "dateline_or_author": "[...]",
  "body_text": "[...]"
 }
]
Final Verification Step:
Before generating your final output, mentally review the body_text of each extracted article. If the text suddenly changes subject entirely (e.g., shifting from local politics to a sports match), you have likely mixed two articles. Fix the boundaries before outputting.""")

  model = "gemini-3.1-pro-preview"
  contents = [
    types.Content(
      role="user",
      parts=[
        msg1_document1,
        msg1_text1
      ]
    ),
  ]
  tools = [
    types.Tool(google_search=types.GoogleSearch()),
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 1,
    top_p = 0.95,
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )],
    tools = tools,
    thinking_config=types.ThinkingConfig(
      thinking_level="HIGH",
    ),
  )

  response = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
  )
  #print(response.text)
  return response.text

In [ ]:
response_json = format_json_response(generate("gs://sjangbahadur_veo/DB/5BHOPAL CITY-PG4-0 (1).pdf"))

```json
[
 {
  "article_id": 1,
  "page_number": "4",
  "headline": "औपनिवेशिक मानसिकता से जड़ों की ओर बढ़ते देश की कहानी बयां करती प्रशांत पोल की किताब 'इंडिया से भारत: एक प्रवास'",
  "sub_headline": "पुस्तक विमोचन • प्रज्ञा प्रवाह के कार्यक्रम में मुख्यमंत्री डॉ. मोहन यादव ने किया नई पुस्तक का विमोचन",
  "dateline_or_author": "सिटी रिपोर्टर | भोपाल",
  "body_text": "प्रज्ञा प्रवाह द्वारा आयोजित कार्यक्रम में गोविंद लेखक प्रशांत पोल द्वारा लिखित किताब 'इंडिया से भारत: एक प्रवास' का विमोचन मंगलवार को कुशाभाऊ ठाकरे इंटरनेशनल कंवेंशन सेंटर हॉल में हुआ। लगभग 3 महीनों में लिखी गई इस किताब का विमोचन मुख्यमंत्री डॉ. मोहन यादव और अन्य अतिथियों ने किया। कार्यक्रम में मंत्रिमंडल के अनेक सदस्य उपस्थित थे। इसमें पहले अहमदाबाद में किताब का गुजराती भाषा में विमोचन हो चुका है। किताब की प्रस्तावना आरएसएस सरसंघचालक मोहन भागवत ने लिखी है। लेखक प्रशांत के अनुसार, कभी जगदगुरू के सामने समृद्ध देशों में था। लगभग 10वीं शताब्दी तक विश्व जीडीपी का करीब 30% हिस्सा भारत के पास था। लेकिन इस्लामी आक्रांताओं और फि

In [ ]:
def generate_meta_data(news_article):

  msg1_text1 = types.Part.from_text(text="""Role: Act as an expert SEO specialist and digital news editor.Task: I am going to provide you with the text of a news article. Please read the article carefully and generate the following metadata for it, ensuring it is optimized for search engines and maximizes click-through rates (CTR).CRITICAL INSTRUCTION: You must provide your response strictly in valid JSON format. Do not include any introductory text, explanations, or markdown formatting outside of the JSON object.Output Format: Please provide the output using the exact JSON structure below:
```json {
"meta_description": "Write a compelling summary of the Article Text that encourages readers to click. (Limit: 150-160 characters)",
"primary_keyword": "List 1 primary keyword here",
"secondary_keywords": [
"secondary keyword 1",
"secondary keyword 2",
"secondary keyword 3"
],
"tags": [
"Category 1",
"Category 2",
"Category 3"
],
"summmary": "A short summary of the Article Text"
}```
Article Text:""")
  msg1_text2 = types.Part.from_text(text=news_article)

  model = "gemini-3.1-pro-preview"
  contents = [
    types.Content(
      role="user",
      parts=[
        msg1_text1,
        msg1_text2
      ]
    ),
  ]
  tools = [
    types.Tool(google_search=types.GoogleSearch()),
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 1,
    top_p = 0.95,
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )],
    tools = tools,
    thinking_config=types.ThinkingConfig(
      thinking_level="HIGH",
    ),
  )

  response = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
  )
  #print(response.text)
  return response.text

In [ ]:
for article in response_json:
  print(f"Generating and updating metadata for Article ID: {article['article_id']}")
  metadata_output = generate_meta_data(json.dumps(article))
  article['metadata'] = format_json_response(metadata_output)
  print(f"Metadata for article {article['article_id']} updated.\n---\n")
print("Metadata generation and update complete for all articles.")

Generating and updating metadata for Article ID: 1
{
"meta_description": "भोपाल में सीएम मोहन यादव ने प्रशांत पोल की किताब 'इंडिया से भारत: एक प्रवास' का विमोचन किया। यह पुस्तक देश की जड़ों और खोए इतिहास की कहानी बयां करती है।",
"primary_keyword": "इंडिया से भारत: एक प्रवास",
"secondary_keywords": [
"प्रशांत पोल की किताब",
"सीएम मोहन यादव",
"पुस्तक विमोचन भोपाल"
],
"tags": [
"Book Release",
"Madhya Pradesh",
"Literature",
"History"
],
"summmary": "मुख्यमंत्री डॉ. मोहन यादव ने भोपाल में प्रशांत पोल द्वारा लिखित पुस्तक 'इंडिया से भारत: एक प्रवास' का विमोचन किया। आरएसएस प्रमुख मोहन भागवत की प्रस्तावना वाली यह किताब औपनिवेशिक मानसिकता से मुक्त होकर देश के अपनी सांस्कृतिक और ऐतिहासिक जड़ों की ओर लौटने के सफर को दर्शाती है।"
}
Metadata for article 1 updated.
---

Generating and updating metadata for Article ID: 2
{
"meta_description": "भोपाल शहर की प्रमुख खबरें: सेवा भारती का स्वास्थ्य शिविर, साधु समाज की निगम मंडलों में मांग और शांति ओचक कैंट में अनियमितताएं। ताज़ा हलचल और अपडेट पढ़ें।",
"p

In [ ]:
print(json.dumps(response_json, indent=2, ensure_ascii=False))

[
  {
    "article_id": 1,
    "page_number": "4",
    "headline": "औपनिवेशिक मानसिकता से जड़ों की ओर बढ़ते देश की कहानी बयां करती प्रशांत पोल की किताब 'इंडिया से भारत: एक प्रवास'",
    "sub_headline": "पुस्तक विमोचन • प्रज्ञा प्रवाह के कार्यक्रम में मुख्यमंत्री डॉ. मोहन यादव ने किया नई पुस्तक का विमोचन",
    "dateline_or_author": "सिटी रिपोर्टर | भोपाल",
    "body_text": "प्रज्ञा प्रवाह द्वारा आयोजित कार्यक्रम में गोविंद लेखक प्रशांत पोल द्वारा लिखित किताब 'इंडिया से भारत: एक प्रवास' का विमोचन मंगलवार को कुशाभाऊ ठाकरे इंटरनेशनल कंवेंशन सेंटर हॉल में हुआ। लगभग 3 महीनों में लिखी गई इस किताब का विमोचन मुख्यमंत्री डॉ. मोहन यादव और अन्य अतिथियों ने किया। कार्यक्रम में मंत्रिमंडल के अनेक सदस्य उपस्थित थे। इसमें पहले अहमदाबाद में किताब का गुजराती भाषा में विमोचन हो चुका है। किताब की प्रस्तावना आरएसएस सरसंघचालक मोहन भागवत ने लिखी है। लेखक प्रशांत के अनुसार, कभी जगदगुरू के सामने समृद्ध देशों में था। लगभग 10वीं शताब्दी तक विश्व जीडीपी का करीब 30% हिस्सा भारत के पास था। लेकिन इस्लामी आक्रांताओं 

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-2"

def generate_embedding(summary_text):
    embedding_client = genai.Client(vertexai=True, project=PROJECT_ID, location="us")
    content = types.Content(
    parts=[
        types.Part.from_text(text=summary_text),
    ]
    )
    result = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[content],
    )
    return result.embeddings[0].values

for article in response_json:
    if 'metadata' in article and 'summmary' in article['metadata']:
        summary_text = article['metadata']['summmary']
        print(f"Generating embedding for Article ID: {article['article_id']}")
        embedding = generate_embedding(summary_text)
        article['metadata']['embedding'] = embedding
        print(f"Embedding for article {article['article_id']} generated and added.")
    else:
        print(f"Article ID: {article['article_id']} does not have a summary for embedding.")
print("\nEmbedding generation complete for all articles.")

Generating embedding for Article ID: 1
Embedding for article 1 generated and added.
Generating embedding for Article ID: 2
Embedding for article 2 generated and added.
Generating embedding for Article ID: 3
Embedding for article 3 generated and added.
Generating embedding for Article ID: 4
Embedding for article 4 generated and added.
Generating embedding for Article ID: 5
Embedding for article 5 generated and added.
Generating embedding for Article ID: 6
Embedding for article 6 generated and added.
Generating embedding for Article ID: 7
Embedding for article 7 generated and added.
Generating embedding for Article ID: 8
Embedding for article 8 generated and added.
Generating embedding for Article ID: 9
Embedding for article 9 generated and added.
Generating embedding for Article ID: 10
Embedding for article 10 generated and added.

Embedding generation complete for all articles.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --- Cosine Similarity Search (in Python) ---
print("--- Cosine Similarity Search (in Python) ---")

def find_similar_articles(query_text, articles_data, top_n=5):
    """
    Finds articles most similar to the query based on cosine similarity of embeddings.

    Args:
        query_text (str): The search query.
        articles_data (list): A list of article dictionaries, each with 'metadata' and 'embedding'.
        top_n (int): The number of top similar articles to return.

    Returns:
        list: A list of (article, similarity_score) tuples, sorted by similarity.
    """
    query_embedding = generate_embedding(query_text)

    similarities = []
    for article in articles_data:
        # Ensure the article has metadata and an embedding
        if 'metadata' in article and 'embedding' in article['metadata'] and article['metadata']['embedding'] is not None:
            article_embedding = np.array(article['metadata']['embedding']).reshape(1, -1)
            query_embedding_np = np.array(query_embedding).reshape(1, -1)
            similarity = cosine_similarity(query_embedding_np, article_embedding)[0][0]
            similarities.append((article, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

search_query = "शादी के नए ट्रेंड और बुकिंग की स्थिति"  # @param {type: "string", placeholder: "Search Query", isTemplate: true}
similar_articles = find_similar_articles(search_query, response_json, top_n=3)

print(f"Top 3 similar articles for query: '{search_query}'")
for article, score in similar_articles:
    print(f"  Article ID: {article['article_id']}, Headline: {article['headline']}, Similarity: {score:.4f}")
    if 'metadata' in article and 'summmary' in article['metadata']:
        print(f"    Summary: {article['metadata']['summmary']}")
    print("-" * 20)



--- Cosine Similarity Search (in Python) ---
Top 3 similar articles for query: 'शादी के नए ट्रेंड और बुकिंग की स्थिति'
  Article ID: 10, Headline: शहनाइयों पर 14 मई के बाद 'ब्रेक'... 9 दिन में होंगी 1500 शादियां; मैरिज गार्डन 90% तक बुक, Similarity: 0.6938
    Summary: भोपाल में 14 मई के बाद शुभ विवाह मुहूर्तों पर विराम लगने जा रहा है। इसके चलते आगामी 9 दिनों में शहर में करीब 1500 शादियां होने का अनुमान है। शहर और आसपास के 90% मैरिज गार्डन बुक हो चुके हैं। इस सीजन में मिनी वेडिंग, इको-फ्रेंडली सजावट और थीम बेस्ड इवेंट्स का ट्रेंड बढ़ गया है, हालांकि महंगाई के कारण कैटरिंग और डेकोरेशन की बुकिंग 10-15% तक महंगी हो गई है।
--------------------
  Article ID: 4, Headline: 21 किलो मोगरे से हनुमान का शृंगार, दीपकों से आरती, सुंदरकांड पाठ हुआ, Similarity: 0.5123
    Summary: भोपाल में अवध की ऐतिहासिक परंपरा का पालन करते हुए ज्येष्ठ माह के पहले मंगलवार को 'बड़ा मंगल' और संकष्टी चतुर्थी मनाई गई। इस अवसर पर शहर के विभिन्न मंदिरों में 21 किलो मोगरे से हनुमान जी का श्रृंगार, 5100 दीपकों से आरती, और 

### Not tested the BQ part yet

In [ ]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

# --- BigQuery Workflow ---
print("\n--- BigQuery Workflow ---")

# BigQuery Client and Table Setup
# PROJECT_ID and LOCATION are expected to be available from previous cells.
bq_client = bigquery.Client(project=PROJECT_ID)
dataset_id = "news_articles_dataset"
table_id = "article_embeddings_table"
table_ref = bq_client.dataset(dataset_id).table(table_id)

# Create Dataset if it doesn't exist
try:
    bq_client.get_dataset(dataset_id)
    print(f"Dataset '{dataset_id}' already exists.")
except NotFound:
    dataset = bigquery.Dataset(bq_client.dataset(dataset_id))
    dataset.location = "US" # Or your desired location, e.g., 'asia-south1' if that's where Vertex AI was used
    dataset = bq_client.create_dataset(dataset, timeout=30)
    print(f"Dataset '{dataset_id}' created.")

# Define Table Schema
schema = [
    bigquery.SchemaField("article_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("page_number", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("headline", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("sub_headline", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("dateline_or_author", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("body_text", "STRING", mode="NULLABLE"),
    # Metadata fields from the nested dictionary
    bigquery.SchemaField("meta_description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("primary_keyword", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("secondary_keywords", "STRING", mode="REPEATED"), # Store as STRING array
    bigquery.SchemaField("tags", "STRING", mode="REPEATED"), # Store as STRING array
    bigquery.SchemaField("summary", "STRING", mode="NULLABLE"), # Corrected from 'summmary'
    bigquery.SchemaField("embedding", "FLOAT", mode="REPEATED"), # This is the vector embedding
]

# Create Table if it doesn't exist and load data
try:
    bq_client.get_table(table_ref)
    print(f"Table '{table_id}' already exists. Skipping data insertion to avoid duplicates.")
    # For a real application, you might want to update or merge data.
except NotFound:
    print(f"Creating table '{table_id}' and inserting data.")
    table = bigquery.Table(table_ref, schema=schema)
    table = bq_client.create_table(table)
    print(f"Table '{table_id}' created successfully.")

    # Prepare data for insertion
    rows_to_insert = []
    for article in response_json:
        if 'metadata' in article and 'embedding' in article['metadata'] and article['metadata']['embedding'] is not None:
            # Flatten the structure for BigQuery insertion
            row = {
                "article_id": article.get("article_id"),
                "page_number": article.get("page_number"),
                "headline": article.get("headline"),
                "sub_headline": article.get("sub_headline"),
                "dateline_or_author": article.get("dateline_or_author"),
                "body_text": article.get("body_text"),
                "meta_description": article['metadata'].get("meta_description"),
                "primary_keyword": article['metadata'].get("primary_keyword"),
                "secondary_keywords": article['metadata'].get("secondary_keywords", []), # Default to empty list
                "tags": article['metadata'].get("tags", []), # Default to empty list
                "summary": article['metadata'].get("summmary"), # Using 'summmary' from original data
                "embedding": article['metadata'].get("embedding")
            }
            rows_to_insert.append(row)

    if rows_to_insert:
        errors = bq_client.insert_rows_json(table_ref, rows_to_insert)
        if errors:
            print(f"Encountered errors while inserting rows: {errors}")
        else:
            print(f"Successfully inserted {len(rows_to_insert)} rows into BigQuery table '{table_id}'.")
    else:
        print("No articles with embeddings found to insert into BigQuery.")

# --- Inline Similarity Search on BigQuery ---
print("\n--- Inline Similarity Search on BigQuery ---")

bq_search_query_text = "शादी की तैयारी कैसे करें" # Example query for BigQuery similarity search
bq_query_embedding = generate_embedding(bq_search_query_text)

# Convert embedding list to a string representation for SQL array literal
bq_query_embedding_str = '[' + ','.join(map(str, bq_query_embedding)) + ']'

# BigQuery SQL query to perform cosine similarity calculation
sql_query = f"""
SELECT
    t.article_id,
    t.headline,
    t.summary,
    (
        SELECT SUM(query_val * article_val)
        FROM UNNEST(ARRAY{bq_query_embedding_str}) AS query_val WITH OFFSET q_idx
        JOIN UNNEST(t.embedding) AS article_val WITH OFFSET a_idx
        ON q_idx = a_idx
    ) / (
        SQRT((SELECT SUM(val*val) FROM UNNEST(ARRAY{bq_query_embedding_str}) AS val)) *
        SQRT((SELECT SUM(val*val) FROM UNNEST(t.embedding) AS val))
    ) AS cosine_similarity_score
FROM
    `{PROJECT_ID}.{dataset_id}.{table_id}` AS t
WHERE
    ARRAY_LENGTH(t.embedding) > 0 -- Ensure embeddings exist
ORDER BY
    cosine_similarity_score DESC
LIMIT 5
"""

print(f"Executing BigQuery similarity search for query: '{bq_search_query_text}'")
query_job = bq_client.query(sql_query)
results = query_job.result()

print("Top 5 similar articles from BigQuery:")
for row in results:
    print(f"  Article ID: {row.article_id}, Headline: {row.headline}, Summary: {row.summary}, Similarity: {row.cosine_similarity_score:.4f}")

print("\nBigQuery workflow complete.")



--- BigQuery Workflow ---
Dataset 'news_articles_dataset' created.
Creating table 'article_embeddings_table' and inserting data.
Table 'article_embeddings_table' created successfully.
Successfully inserted 10 rows into BigQuery table 'article_embeddings_table'.

--- Inline Similarity Search on BigQuery ---
Executing BigQuery similarity search for query: 'शादी की तैयारी कैसे करें'
Top 5 similar articles from BigQuery:
  Article ID: 10, Headline: शहनाइयों पर 14 मई के बाद 'ब्रेक'... 9 दिन में होंगी 1500 शादियां; मैरिज गार्डन 90% तक बुक, Summary: भोपाल में 14 मई के बाद शुभ विवाह मुहूर्तों पर विराम लगने जा रहा है। इसके चलते आगामी 9 दिनों में शहर में करीब 1500 शादियां होने का अनुमान है। शहर और आसपास के 90% मैरिज गार्डन बुक हो चुके हैं। इस सीजन में मिनी वेडिंग, इको-फ्रेंडली सजावट और थीम बेस्ड इवेंट्स का ट्रेंड बढ़ गया है, हालांकि महंगाई के कारण कैटरिंग और डेकोरेशन की बुकिंग 10-15% तक महंगी हो गई है।, Similarity: 0.5721
  Article ID: 4, Headline: 21 किलो मोगरे से हनुमान का शृंगार, दीपकों से आ